# Modelado Predictivo del Módulo de Young ($E$ en GPa) en Aleaciones MPEAs
## Cuaderno de Experimentación, Optimización y Selección de Modelos

En este cuaderno se desarrolla el pipeline integral de modelado predictivo para el **Módulo de Young Calculado (`Calculated Young modulus (GPa)`)** en Aleaciones de Múltiples Elementos Principales (*Multi-Principal Element Alloys*, MPEAs).

El módulo de elasticidad o de Young refleja la rigidez mecánica y la intensidad de los enlaces interatómicos en el retículo cristalino. Al tratarse de aleaciones de alta entropía con múltiples elementos principales, los descriptores termodinámicos y electrónicos son altamente determinantes.

### Flujo Metodológico:
1. **Configuración del Entorno y Carga de Datos**: Detección y lectura local de la matriz numérica featurizada (`dataset_final_ml_2.csv` / `dataset_final_ml.csv`).
2. **Partición de Datos**: División reproducible 80% entrenamiento y 20% prueba con semilla fija (`random_state=42`).
3. **Benchmarking de Modelos**: Comparativa entre *Random Forest Regressor* (Bagging) y *XGBoost Regressor* (Gradient Boosting).
4. **Análisis de Explicabilidad e Importancia de Variables**: Consenso ponderado de importancia física (promedio RF + XGBoost) y visualización del Top 15.
5. **Reducción Dimensional (Barrido Grueso)**: Análisis de sensibilidad de 10 a 100 descriptores para verificar la velocidad de convergencia del error.
6. **Optimización Fina (Micro-Barrido)**: Zoom de alta precisión (15 a 39 variables) para localizar el *Punto Dulce* exacto.
7. **Entrenamiento Final y Serialización**: Ajuste del modelo de producción con 27 variables y exportación de archivos binarios (`.pkl`) y metadata estandarizada (`.json`).

## Paso 1: Configuración de Librerías y Compatibilidad de Entorno

Importamos las bibliotecas científicas requeridas para el flujo de trabajo en Machine Learning. La configuración está preparada para ejecutarse localmente sin dependencias obligatorias de la nube.

In [ ]:
import os
import json
import joblib
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor

# Compatibilidad con Google Colab (opcional si se ejecuta en la nube)
try:
    from google.colab import drive
    # drive.mount('/content/drive')
except ImportError:
    pass

## Paso 2: Carga de la Matriz Numérica y Partición de Datos (Train/Test Split)

Se carga la matriz estructurada para el Módulo de Young (550 muestras x 271 columnas numéricas), resultante de la curación con $k=13$ headers y la extracción de descriptores composicionales Magpie, soluciones sólidas de Yang y orbitales de valencia.

Se separa la propiedad objetivo `Calculated Young modulus (GPa)` de las 270 características de entrada ($X$), y se aplica la partición 80/20 con semilla fija.

In [ ]:
# 1. Cargar la matriz matemática desde el entorno local
posibles_rutas_csv = [
    Path("dataset_final_ml_2.csv"),
    Path("dataset_final_ml.csv"),
    Path("Modulo_de_young") / "dataset_final_ml_2.csv",
    Path("Modulo_de_young") / "dataset_final_ml.csv",
    Path("..") / "Modulo_de_young" / "dataset_final_ml_2.csv",
    Path("..") / "Modulo_de_young" / "dataset_final_ml.csv"
]
ruta_csv = next((p for p in posibles_rutas_csv if p.is_file()), None)

if ruta_csv is None:
    raise FileNotFoundError(
        "No se encontró 'dataset_final_ml_2.csv' ni 'dataset_final_ml.csv'. Verifique el directorio de ejecución."
    )

print(f"Cargando dataset desde: {ruta_csv.resolve()}")
df = pd.read_csv(ruta_csv)
print(f"Dimensiones de la matriz cargada: {df.shape[0]} filas x {df.shape[1]} columnas")

# 2. Separar características predictoras (X) y variable objetivo (y)
variable_objetivo = 'Calculated Young modulus (GPa)'

X = df.drop(columns=[variable_objetivo])
y = df[variable_objetivo]

# 3. Partición de datos: 80% entrenamiento y 20% prueba (random_state=42 fijo para reproducibilidad)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Muestras para entrenamiento (80%): {X_train.shape[0]} | Muestras para prueba (20%): {X_test.shape[0]}")

## Paso 3: Definición e Instanciación de Modelos de Regresión

Instanciamos los dos regresores de ensamble para comparar su comportamiento frente a la rigidez elástica:
1. **Random Forest Regressor**: Ensamble paralelo de 100 árboles de decisión independientes.
2. **XGBoost Regressor**: Algoritmo de boosting optimizado con tasa de aprendizaje `learning_rate=0.1`.

In [ ]:
# 4. Instanciar los modelos de regresión
# Se utiliza n_jobs=-1 para paralelizar el cómputo en todos los núcleos de CPU disponibles
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)

## Paso 4: Entrenamiento de los Modelos Base

Se ajustan ambos modelos sobre el 80% de los datos (`X_train`, `y_train`).

In [ ]:
# 5. Entrenamiento
print("Entrenando Random Forest Regressor...")
rf_model.fit(X_train, y_train)

print("Entrenando XGBoost Regressor...")
xgb_model.fit(X_train, y_train)
print("Entrenamiento completado exitosamente.")

## Paso 5: Predicciones y Evaluación de Rendimiento

Se evalúan las predicciones sobre el 20% de prueba (`X_test`) calculando $R^2$ y $RMSE$ (en Gigapascales, GPa).

In [ ]:
# 6. Predicciones sobre el conjunto de prueba (20%)
rf_pred = rf_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)

# 7. Evaluación de métricas
rf_r2 = r2_score(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
xgb_r2 = r2_score(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))

print("
=== Resultados del Modelo: Módulo de Young (GPa) ===")
print(f"Random Forest -> R2: {rf_r2:.4f} | RMSE: {rf_rmse:.2f} GPa")
print(f"XGBoost       -> R2: {xgb_r2:.4f} | RMSE: {xgb_rmse:.2f} GPa")

## Paso 6: Explicabilidad y Consenso de Importancia de Variables (Feature Importance)

Se calcula la importancia relativa de cada descriptor en ambos algoritmos y se obtiene el **consenso promedio**.

Esto revela los 15 descriptores más determinantes para el Módulo de Young:
- Electronegatividad media de Magpie (fuerza de enlace atómico).
- Descriptores de orbitales de valencia (fracción de electrones s, p, d).
- Densidad teórica calculada (`Calculated Density (g/cm^3)`).
- Temperatura de fusión máxima y peso atómico.
- Contenido de elementos refractarios y endurecedores ($Cr$, $Mo$).

In [ ]:
# 1. Extraer importancias de ambos modelos
df_importancias = pd.DataFrame({
    'Variable': X.columns,
    'Importancia_RF': rf_model.feature_importances_,
    'Importancia_XGB': xgb_model.feature_importances_
})

# 2. Calcular el consenso (promedio) para definir el Top 15
df_importancias['Importancia_Promedio'] = df_importancias[['Importancia_RF', 'Importancia_XGB']].mean(axis=1)
top_15_consenso = df_importancias.sort_values(by='Importancia_Promedio', ascending=False).head(15)

# 3. Reestructurar la tabla para visualización comparativa con Seaborn
df_grafico = top_15_consenso.melt(
    id_vars='Variable',
    value_vars=['Importancia_RF', 'Importancia_XGB'],
    var_name='Modelo',
    value_name='Nivel de Importancia'
)

# 4. Generar el gráfico comparativo
plt.figure(figsize=(12, 8))
sns.barplot(x='Nivel de Importancia', y='Variable', hue='Modelo', data=df_grafico, palette='Set2')
plt.title('Intersección de Variables Físicas: Random Forest vs XGBoost (Módulo de Young)', fontsize=14)
plt.xlabel('Peso / Nivel de Importancia en el Modelo', fontsize=12)
plt.ylabel('Descriptores Físico-Químicos', fontsize=12)
plt.tight_layout()
plt.show()

## Paso 7: Reducción Dimensional - Barrido Grueso (10 a 100 Variables)

Evaluamos el comportamiento del error cuadrático medio ($RMSE$) reteniendo desde 10 hasta 100 variables en incrementos de 10.

A diferencia del límite elástico, el módulo de Young presenta una estabilización de error muy acelerada gracias a la fuerte correlación física entre los parámetros atómicos elementales y la rigidez elástica.

In [ ]:
# Extraer importancias del modelo base XGBoost
importancias_xgb = xgb_model.feature_importances_
indices_ordenados = np.argsort(importancias_xgb)[::-1]

# Barrido grueso: de 10 a 100 variables, en pasos de 10
rango_variables = list(range(10, 110, 10))

errores_xgb = []
errores_rf = []

print("Calculando la curva de rendimiento y el valle de error...")

for n_vars in rango_variables:
    columnas_top = X.columns[indices_ordenados[:n_vars]]
    X_train_red = X_train[columnas_top]
    X_test_red = X_test[columnas_top]

    # Entrenar modelos rápidos
    xgb_tmp = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
    rf_tmp = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

    xgb_tmp.fit(X_train_red, y_train)
    rf_tmp.fit(X_train_red, y_train)

    # Registrar el RMSE en el conjunto de prueba
    errores_xgb.append(np.sqrt(mean_squared_error(y_test, xgb_tmp.predict(X_test_red))))
    errores_rf.append(np.sqrt(mean_squared_error(y_test, rf_tmp.predict(X_test_red))))

# Graficar el comportamiento
plt.figure(figsize=(10, 6))
plt.plot(rango_variables, errores_xgb, marker='o', linewidth=2, color='coral', label='XGBoost')
plt.plot(rango_variables, errores_rf, marker='s', linewidth=2, color='teal', label='Random Forest')

# Marcar zona óptima estimada
plt.axvline(x=30, color='gray', linestyle='--', alpha=0.7, label='Zona Óptima Estimada (~25-30 Vars)')

plt.title('Curva de Rendimiento: Número de Variables vs RMSE (Módulo de Young)', fontsize=14)
plt.xlabel('Cantidad de Variables Retenidas (Top N)', fontsize=12)
plt.ylabel('Error RMSE (GPa) - Menor es mejor', fontsize=12)
plt.gca().invert_xaxis()
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Paso 8: Micro-Barrido Fino - Localización del Punto Dulce (15 a 39 Variables)

Se efectúa un micro-escaneo de resolución unitaria explorando entre 15 y 39 variables para ubicar con exactitud matemática el punto de menor error y máxima parsimonia.

In [ ]:
# Micro-barrido: iterar de 15 a 39, de 1 en 1
rango_fino = list(range(15, 40, 1))
errores_micro = []

print("Iniciando escáner de alta precisión con lupa gráfica (15 a 39 variables)...")

for n_vars in rango_fino:
    columnas_top = X.columns[indices_ordenados[:n_vars]]
    X_train_red = X_train[columnas_top]
    X_test_red = X_test[columnas_top]

    xgb_tmp = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
    xgb_tmp.fit(X_train_red, y_train)

    rmse_actual = np.sqrt(mean_squared_error(y_test, xgb_tmp.predict(X_test_red)))
    errores_micro.append(rmse_actual)

# Encontrar al ganador absoluto
mejor_idx = np.argmin(errores_micro)
mejor_n = rango_fino[mejor_idx]
mejor_rmse = errores_micro[mejor_idx]

print("
=== El Punto Dulce Absoluto ===")
print(f"Número óptimo de columnas a retener: {mejor_n}")
print(f"RMSE mínimo alcanzable en este valle: {mejor_rmse:.2f} GPa")

# Gráfico de alta resolución
plt.figure(figsize=(10, 5))
plt.plot(rango_fino, errores_micro, marker='o', linestyle='-', color='coral', linewidth=2, markersize=6)

# Marcar el punto óptimo
plt.plot(mejor_n, mejor_rmse, marker='*', markersize=18, color='red', label=f'Punto Dulce: {mejor_n} vars ({mejor_rmse:.2f} GPa)')
plt.axvline(x=mejor_n, color='crimson', linestyle='--', alpha=0.5)

plt.title('Micro-Barrido XGBoost: Zoom al Valle del Error (15-39 Variables)', fontsize=14)
plt.xlabel('Cantidad de Variables Retenidas', fontsize=12)
plt.ylabel('Error RMSE (GPa)', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Paso 9: Entrenamiento del Modelo de Producción y Serialización

El micro-barrido establece que con únicamente **27 variables** se obtiene el modelo óptimo ($R^2 = 0.9969$, $RMSE = 2.53$ GPa):
1. Se filtra el dataset a las 27 características principales.
2. Se entrena el estimador `XGBRegressor` de producción.
3. Se serializa el modelo en `xgboost_Calculated Young modulus (GPa).pkl`.
4. Se exporta la lista de descriptores requeridos y métricas oficiales a `variables_Calculated Young modulus (GPa).json`, permitiendo su consumo inmediato por la interfaz web de inferencia.

In [ ]:
# 0. Localizar carpeta de exportación en entorno local (Models en la raíz o models local)
posibles_carpetas_modelos = [
    Path("../Models"),
    Path("Models"),
    Path("models")
]
carpeta_modelos = next((p for p in posibles_carpetas_modelos if p.is_dir()), None)
if carpeta_modelos is None:
    carpeta_modelos = Path("models")
    carpeta_modelos.mkdir(parents=True, exist_ok=True)

# 1. Fijar el punto dulce descubierto (27 variables)
n_vars = 27
columnas_top_27 = X.columns[indices_ordenados[:n_vars]]

# 2. Construir el dataset de producción ultra-optimizado
X_train_final = X_train[columnas_top_27]
X_test_final = X_test[columnas_top_27]

# 3. Entrenar el modelo Campeón
print(f"Entrenando modelo XGBoost para '{variable_objetivo}' ({n_vars} variables)...")
xgb_final = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_final.fit(X_train_final, y_train)

# 4. Evaluar y calcular el rendimiento final
predicciones_finales = xgb_final.predict(X_test_final)
r2_final = float(r2_score(y_test, predicciones_finales))
rmse_final = float(np.sqrt(mean_squared_error(y_test, predicciones_finales)))

# Imprimir métricas en consola
print("
=== Rendimiento del Modelo de Producción ===")
print(f"Variable Objetivo:       {variable_objetivo}")
print(f"R2 (Varianza Explicada): {r2_final:.4f}")
print(f"RMSE (Error Mínimo):     {rmse_final:.2f} GPa\n")

# 5. Exportar el Modelo serializado
nombre_modelo = carpeta_modelos / f"xgboost_{variable_objetivo}.pkl"
joblib.dump(xgb_final, nombre_modelo)

# 6. Preparar el diccionario con la metadata completa (Métricas + Variables)
lista_variables = columnas_top_27.tolist()

metadata_exportacion = {
    "variable_objetivo": variable_objetivo,
    "rendimiento": {
        "R2": round(r2_final, 4),
        "RMSE": round(rmse_final, 2)
    },
    "cantidad_variables": n_vars,
    "variables_entrada": lista_variables
}

# 7. Exportar como JSON en la carpeta de modelos
nombre_json = carpeta_modelos / f"variables_{variable_objetivo}.json"
with open(nombre_json, "w", encoding="utf-8") as f:
    json.dump(metadata_exportacion, f, indent=4)

# Confirmación de guardado e impresión de variables en consola
print(f"¡Éxito! Modelo exportado como: '{nombre_modelo.resolve()}'")
print(f"¡Metadata exportada como: '{nombre_json.resolve()}'\n")

print(f"=== Variables de Entrada Requeridas ({n_vars}) ===")
for i, var in enumerate(lista_variables, 1):
    print(f"{i}. {var}")